In [1]:
%matplotlib inline

# Process Optimization

First, instantiate an `OptimizationProblem`.

In [2]:
from CADETProcess.optimization import OptimizationProblem
optimization_problem = OptimizationProblem('batch_elution')

[WARNING 03-30 22:19:26] ax.storage.sqa_store.with_db_settings_base: Ax currently requires a sqlalchemy version below 2.0. This will be addressed in a future release. Disabling SQL storage in Ax for now, if you would like to use SQL storage please install Ax with mysql extras via `pip install ax-platform[mysql]`.


Import the fully configured `Process` from the examples and add it to the `OptimizationProblem`.

In [3]:
from examples.batch_elution.process import process
optimization_problem.add_evaluation_object(process)

Optimize the cycle time and feed duration.

In [4]:
optimization_problem.add_variable('cycle_time', lb=10, ub=600)
optimization_problem.add_variable('feed_duration.time', lb=10, ub=300)

OptimizationVariable(name=feed_duration.time, evaluation_objects=['batch elution'], parameter_path=feed_duration.time, lb=10, ub=300)

Ensure the feed duration is always shorter than the cycle time by adding a linear constraint.

In [5]:
optimization_problem.add_linear_constraint(
    ['feed_duration.time', 'cycle_time'], [1, -1]
)

Configure and register a simulator as an evaluator to ensure cyclic stationarity.

In [6]:
from CADETProcess.simulator import Cadet
process_simulator = Cadet()
process_simulator.evaluate_stationarity = True

optimization_problem.add_evaluator(process_simulator)

Configure and register the fractionation optimizer as another evaluator.

In [7]:
from CADETProcess.fractionation import FractionationOptimizer
frac_opt = FractionationOptimizer()

optimization_problem.add_evaluator(
    frac_opt,
    kwargs={
        'purity_required': [0.95, 0.95],
        'ignore_failed': False,
        'allow_empty_fractions': False,
    }
)

Define the objectives.

In [8]:
from CADETProcess.performance import PerformanceProduct
ranking = [1, 1]
performance = PerformanceProduct(ranking=ranking)

optimization_problem.add_objective(
    performance,
    requires=[process_simulator, frac_opt],
    minimize=False,
)

Add a callback for post-processing.

In [9]:
def callback(fractionation, individual, evaluation_object, callbacks_dir):
    fractionation.plot_fraction_signal(
        file_name=f'{callbacks_dir}/{individual.id}_{evaluation_object}_fractionation.png',
    )

optimization_problem.add_callback(
    callback, requires=[process_simulator, frac_opt]
)

Configure the optimizer using `U_NSGA3`.

In [10]:
from CADETProcess.optimization import U_NSGA3
optimizer = U_NSGA3()

optimizer.n_cores = 8
optimizer.pop_size = 32
optimizer.n_max_gen = 16

In [11]:
results = optimizer.optimize(
    optimization_problem,
    use_checkpoint=False,
)

[WARNING 03-30 22:19:38] ax.storage.sqa_store.with_db_settings_base: Ax currently requires a sqlalchemy version below 2.0. This will be addressed in a future release. Disabling SQL storage in Ax for now, if you would like to use SQL storage please install Ax with mysql extras via `pip install ax-platform[mysql]`.
[WARNING 03-30 22:19:38] ax.storage.sqa_store.with_db_settings_base: Ax currently requires a sqlalchemy version below 2.0. This will be addressed in a future release. Disabling SQL storage in Ax for now, if you would like to use SQL storage please install Ax with mysql extras via `pip install ax-platform[mysql]`.


[WARNING 03-30 22:19:38] ax.storage.sqa_store.with_db_settings_base: Ax currently requires a sqlalchemy version below 2.0. This will be addressed in a future release. Disabling SQL storage in Ax for now, if you would like to use SQL storage please install Ax with mysql extras via `pip install ax-platform[mysql]`.


[WARNING 03-30 22:19:39] ax.storage.sqa_store.with_db_settings_base: Ax currently requires a sqlalchemy version below 2.0. This will be addressed in a future release. Disabling SQL storage in Ax for now, if you would like to use SQL storage please install Ax with mysql extras via `pip install ax-platform[mysql]`.
[WARNING 03-30 22:19:40] ax.storage.sqa_store.with_db_settings_base: Ax currently requires a sqlalchemy version below 2.0. This will be addressed in a future release. Disabling SQL storage in Ax for now, if you would like to use SQL storage please install Ax with mysql extras via `pip install ax-platform[mysql]`.
[WARNING 03-30 22:19:40] ax.storage.sqa_store.with_db_settings_base: Ax currently requires a sqlalchemy version below 2.0. This will be addressed in a future release. Disabling SQL storage in Ax for now, if you would like to use SQL storage please install Ax with mysql extras via `pip install ax-platform[mysql]`.


[WARNING 03-30 22:19:40] ax.storage.sqa_store.with_db_settings_base: Ax currently requires a sqlalchemy version below 2.0. This will be addressed in a future release. Disabling SQL storage in Ax for now, if you would like to use SQL storage please install Ax with mysql extras via `pip install ax-platform[mysql]`.


[WARNING 03-30 22:19:41] ax.storage.sqa_store.with_db_settings_base: Ax currently requires a sqlalchemy version below 2.0. This will be addressed in a future release. Disabling SQL storage in Ax for now, if you would like to use SQL storage please install Ax with mysql extras via `pip install ax-platform[mysql]`.


...

Evaluation of PerformanceProduct failed at [59.68712392 48.75952616] with Error 'No areas found with sufficient purity for component(s) ['B'].'. Returning bad metrics.


Evaluation of PerformanceProduct failed at [128.37617107  46.94244976] with Error 'No areas found with sufficient purity for component(s) ['A'].'. Returning bad metrics.


    16 |      512 |      1 |  0.000000E+00 |             f


Finished Generation 16.


x: [163.79594219  48.73933326], f: [0.09399485]
